# 02 — Exploratory Data Analysis (EDA)
**Goal:** Explore national-team performance, match scoring trends, and source completeness. Run `01_data_cleaning.ipynb` first. All visualizations are descriptive; no prediction model is trained here.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data' / 'raw').exists():
    raise FileNotFoundError('Open this notebook with working directory at the project root or notebooks folder.')
sys.path.insert(0, str(PROJECT_ROOT))
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_REPORTS = PROJECT_ROOT / 'data' / 'reports'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_REPORTS.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
matches=pd.read_csv(DATA_PROCESSED/'matches_clean.csv',parse_dates=['date'])
standings=pd.read_csv(DATA_PROCESSED/'team_year_results_90.csv')
summary=pd.read_csv(DATA_PROCESSED/'tournament_summary_clean.csv')
print('Match years:', matches['year'].min(), '-', matches['year'].max())
print('Matches:',len(matches),'Team-year standings:',len(standings))


## A. Match and goal trends by tournament


In [ ]:
valid=matches[matches['goals_1'].notna()&matches['goals_2'].notna()].copy()
valid['total_goals']=valid['goals_1']+valid['goals_2']
trend=valid.groupby('year').agg(matches=('match_key','count'),
                                goals=('total_goals','sum'),
                                avg_goals=('total_goals','mean')).reset_index()
display(trend.tail(8))
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].plot(trend['year'],trend['matches'],marker='o')
axes[0].set(title='Recorded World Cup matches',xlabel='Tournament year',ylabel='Matches')
axes[1].plot(trend['year'],trend['avg_goals'],marker='o')
axes[1].set(title='Average total goals per match',xlabel='Tournament year',ylabel='Goals')
plt.tight_layout();plt.show()


## B. Recent editions (2002–2026): match outcome mix
We use `ml_eligible` matches with interpretable 90-minute scores; excluded ambiguous knockout scores can change a year’s exact counts.


In [ ]:
modern=matches[(matches['year']>=2002)&matches['ml_eligible'].fillna(False)].copy()
outcomes=pd.crosstab(modern['year'],modern['result_90'])
display(outcomes)
outcomes.plot(kind='bar',stacked=True,figsize=(10,4),title='90-minute results (2002 onward)')
plt.xlabel('Year');plt.ylabel('Matches');plt.tight_layout();plt.show()


## C. Team comparisons using match outcomes
Unlike official standings, these statistics are calculated directly from match scores. Choose a minimum number of games to avoid ranking teams by a single result.


In [ ]:
team_rows=pd.concat([
    valid[['year','date','team_1','goals_1','goals_2']]
       .rename(columns={'team_1':'team','goals_1':'gf','goals_2':'ga'}),
    valid[['year','date','team_2','goals_2','goals_1']]
       .rename(columns={'team_2':'team','goals_2':'gf','goals_1':'ga'})
],ignore_index=True)
team_rows['won']=(team_rows['gf']>team_rows['ga']).astype(int)
recent=team_rows[team_rows['year']>=2002]
team_summary=(recent.groupby('team').agg(
    matches=('won','size'),wins=('won','sum'),
    goals_for=('gf','sum'),goals_against=('ga','sum'))
    .assign(win_rate=lambda d:d['wins']/d['matches'])
    .sort_values(['win_rate','matches'],ascending=False))
minimum_matches=10
leaders=team_summary[team_summary['matches']>=minimum_matches].head(12)
display(leaders)
leaders['win_rate'].sort_values().plot.barh(figsize=(9,5),title='Top win rates (min. 10 games)')
plt.xlabel('Proportion of matches won');plt.tight_layout();plt.show()


## D. Compare two national teams


In [ ]:
team_a='Argentina'
team_b='France'
for team in [team_a,team_b]:
    if team in team_summary.index:
        print(team)
        display(team_summary.loc[[team]])
head_to_head=valid[((valid['team_1']==team_a)&(valid['team_2']==team_b))|
                   ((valid['team_1']==team_b)&(valid['team_2']==team_a))]
display(head_to_head[['year','date','team_1','team_2','goals_1','goals_2','round']])


## E. Inspect data quality and coverage
The observed 90-minute team table is reconstructed from OpenFootball. It is not an official FIFA standings table. Missing knockout scores and tournament formats require attention before building the dashboard.


In [ ]:
coverage=pd.DataFrame({
    'openfootball_matches':matches.groupby('year').size(),
    'teams_with_observed_results':standings.groupby('year').size(),
    'ml_eligible_matches':matches.groupby('year')['ml_eligible'].sum(),
}).fillna(0).astype(int)
display(coverage.tail(10))
print('Nonzero nulls in matches:')
display(matches.isna().sum().loc[lambda s:s>0].sort_values(ascending=False).to_frame('missing'))
coverage.to_csv(DATA_REPORTS/'dataset_coverage.csv')
